# 3) Calling Functions — Exercises

**Goals:** positional vs keyword calls, unpacking (`*`, `**`), higher-order functions (functions as values), callbacks, `key=` usage, lambdas vs named functions, error wrappers.

### Warm-ups

1. **Call with keywords**

```python
def rect(w, h): return w*h
def rect_kw(**kwargs):
    """Call rect using **kwargs expected to have w and h."""
    ...
assert rect_kw(w=3, h=4) == 12
```

2. \**Call with *args**

```python
def hypotenuse(a, b):
    return (a*a + b*b) ** 0.5
def hypot_from_pair(pair):
    """Call hypotenuse using *pair."""
    ...
assert abs(hypot_from_pair((3,4)) - 5) < 1e-9
```

3. **Sort with key**

```python
def sort_by_last_char(words):
    """Return new list sorted by last character; stable; do not mutate input."""
    return sorted(words, key=lambda word: word[-1])
assert sort_by_last_char(["ab","aa","bb"]) == ["aa","ab","bb"]
```

### Core

4. **Apply pipeline**

```python
def apply_pipeline(x, funcs):
    """
    funcs: iterable of callables f: x -> x
    Return result of applying them in order.
    """
    ...
assert apply_pipeline(2, [lambda x: x+3, lambda x: x*10]) == 50
```

5. **Filter + map with callables**

```python
def transform(nums, pred, mapper):
    """Keep items where pred(x) is True, then map with mapper(x)."""
    ...
assert transform([1,2,3,4], lambda x:x%2==0, lambda x:x*x) == [4,16]
```

6. **Group with key function**

```python
def group_by(xs, key_fn):
    """Return dict key->list of items in original order."""
    ...
g = group_by(["aa","b","ccc","dd"], key_fn=len)
assert g == {2:["aa","dd"],1:["b"],3:["ccc"]}
```

7. **Retry wrapper**

```python
def retry(fn, *, attempts=3):
    """
    Call fn(); if it raises, retry up to attempts times.
    Return fn() result or raise last error.
    """
    ...
count = {"n":0}
def flaky():
    count["n"] += 1
    if count["n"] < 2: raise RuntimeError("boom")
    return "ok"
assert retry(flaky, attempts=3) == "ok"
```

8. **Decorator: timeit**

```python
import time
def timeit(fn):
    """Decorator returning (result, elapsed_seconds)."""
    ...
@timeit
def slow_add():
    t0 = time.time()
    while time.time() - t0 < 0.01: pass
    return 42
res, secs = slow_add()
assert res == 42 and secs >= 0
```

9. **Caching (simple memoize)**

```python
def memoize(fn):
    """Decorator caching fn(x) for hashable x."""
    ...
calls = {"n":0}
@memoize
def square(x):
    calls["n"] += 1
    return x*x
assert square(10)==100 and square(10)==100 and calls["n"]==1
```

### Challenge

10. **Compose callables**

```python
def compose(*funcs):
    """
    Return f(x) that applies funcs right-to-left: compose(f,g,h)(x) == f(g(h(x))).
    If no funcs, return identity.
    """
    ...
id_fn = compose()
assert id_fn(5) == 5
f = compose(lambda x:x+1, lambda x:x*2, lambda x:x-3)
assert f(10) == ((10-3)*2)+1
```


In [2]:
def rect(w, h): return w*h
def rect_kw(**kwargs):
    """Call rect using **kwargs expected to have w and h."""
    return kwargs["w"] * kwargs["h"]
assert rect_kw(w=3, h=4) == 12

In [ ]:
def hypotenuse(a, b):
    return (a*a + b*b) ** 0.5
def hypot_from_pair(*pair):
    """Call hypotenuse using *pair."""
    return (pair[0][0]*pair[0][0] + pair[0][1]*pair[0][1]) ** 0.5

assert abs(hypot_from_pair((3,4)) - 5) < 1e-9

In [18]:
def sort_by_last_char(words):
    """Return new list sorted by last character; stable; do not mutate input."""
    return sorted(words, key=lambda word: word[-1])

#sort_by_last_char(["ab","aa","bb"])
assert sort_by_last_char(["ab","aa","bb"]) == ["aa","ab","bb"]

In [21]:
def apply_pipeline(x, funcs):
    """
    funcs: iterable of callables f: x -> x
    Return result of applying them in order.
    """
    for function in funcs:
        x = function(x)
    return x

assert apply_pipeline(2, [lambda x: x+3, lambda x: x*10]) == 50

In [23]:
def transform(nums, pred, mapper):
    """Keep items where pred(x) is True, then map with mapper(x)."""
    return [mapper(num) for num in nums if pred(num)]

assert transform([1,2,3,4], lambda x:x%2==0, lambda x:x*x) == [4,16]

In [25]:
def group_by(xs, key_fn):
    """Return dict key->list of items in original order."""
    results = dict()
    for elements in xs:
        if key_fn(elements) in results:
            results[key_fn(elements)].append(elements)
        else:
            results[key_fn(elements)] = [elements]
    return results

g = group_by(["aa","b","ccc","dd"], key_fn=len)
assert g == {2:["aa","dd"],1:["b"],3:["ccc"]}

In [41]:
def retry(fn, *, attempts=3):
    """
    Call fn(); if it raises, retry up to attempts times.
    Return fn() result or raise last error.
    """
    idx = 1
    while idx <= attempts:
        try:
            results = fn()
            return results
        except Exception as error:
            print(f'Fail number: {idx}')
            idx += 1
    raise error


count = {"n":0}
def flaky():
    count["n"] += 1
    if count["n"] < 2: raise RuntimeError("boom")
    return "ok"
assert retry(flaky, attempts=3) == "ok"

Fail number: 1


In [48]:
import time
def timeit(fn):
    """Decorator returning (result, elapsed_seconds)."""
    def wrapper():
        t0 = time.time()
        result = fn()
        t1 = time.time()
        secs = t1-t0
        return result, secs
    return wrapper

@timeit
def slow_add():
    t0 = time.time()
    while time.time() - t0 < 0.01: pass
    return 42

res, secs = slow_add()
assert res == 42 and secs >= 0

In [50]:
def memoize(fn):
    """Decorator caching fn(x) for hashable x."""
    cache = {}
    def wrapper(x):
        if x in cache:
            return cache[x]
        else:
            result = fn(x)
            cache[x] = result
        return result
    return wrapper


calls = {"n":0}
@memoize
def square(x):
    calls["n"] += 1
    return x*x
assert square(10)==100 and square(10)==100 and calls["n"]==1

In [90]:
def compose(*funcs):
    """
    Return f(x) that applies funcs right-to-left: compose(f,g,h)(x) == f(g(h(x))).
    If no funcs, return identity.
    """
    print(type(funcs))
    def wrapper(x):
        value_out = x
        for idx in range(-1,(len(funcs)+1)*-1, -1):
            put_function = funcs[idx]
            value_out = put_function(value_out)
        return value_out
    return wrapper

id_fn = compose()
assert id_fn(5) == 5
f = compose(lambda x:x+1, lambda x:x*2, lambda x:x-3)
assert f(10) == ((10-3)*2)+1

<class 'tuple'>
<class 'tuple'>
